# 1. Information about the submission

## 1.1 Name and number of the assignment

**Assignment 1 - Research Paper Clustering** (SToP / EMNLP)

## 1.2 Student name

Anastasiia Golubkova

## 1.3 Additional comments

The code of experiments is available in https://github.com/Nastya-golubkova/drift-inspector (fork of thr original one).

# 2. Technical Report

## 2.1 Methodology

### Experimental setup

The goal is to group research papers by topic using their text. I first convert each paper into a numerical representation (an embedding), then cluster papers with similar representations together.

I evaluate the resulting clusters using the official **653 evaluation papers** and their topic labels from the SToP dataset. I report five metrics: **Purity, LRAP, nDCG@1, V-measure, and PairF1**, plus **Coverage** and **noise fraction** for HDBSCAN.

I run five experimental setups. They differ in which papers are used to build the clusters, which parts of each paper are used as input, and which clustering algorithm is applied.

#### 1. Clustering the full collection using abstracts

Here, I use a much larger collection of **6,517 papers** to build the clusters. The input is the abstract of each paper. The 653 evaluation papers are a **subset of this fit collection** (transductive evaluation).

I compare three embedding models: SPECTER2, SemCSE, and **Qwen** (Qwen3-Embedding). I also test different ways of representing long abstracts:

- **R0:** encode the entire abstract at once.
- **R2:** split the abstract into overlapping or consecutive text windows, encode each window, and average the resulting embeddings.
- **R3:** do the same, but give more weight to windows containing more tokens.

After obtaining the embeddings, I use UMAP and HDBSCAN to form clusters. I then evaluate the results on the same 653 official evaluation papers.

The clustering pipeline is in `artifacts/assignment1/pipeline/`.

#### 2. Clustering the full collection using titles and abstracts

I again use all **6,517 papers** (eval subset of fit), but now compare different ways of combining the title and abstract:

- **T0:** use only the title.
- **A1:** concatenate the title and abstract into one text and encode them together.
- **A2:** encode the title and abstract separately, then combine their embeddings using a weight alpha.
- **A3:** combine the similarity scores between papers based on their titles and abstracts, also using alpha.

For A2 and A3, I test several values of alpha: 0.1, 0.2, 0.3, 0.4, and 0.5. This lets us see how much the title and abstract each contribute to the result.

The implementation is in `artifacts/assignment1_title_fusion/run_title_fusion.py`.

#### 3. Clustering a collection of 4,488 papers

I repeat the title-and-abstract experiments from the previous section (T0/A1/A2/A3, plus abstract R0) on a smaller collection of **4,488 papers**. This collection is selected to match the set of papers used by the ACC baseline. The 653 evaluation papers are again a **subset of the fit collection**.

The implementation is in `artifacts/assignment1_fit4488/run_fit4488.py` (also callable via the thin wrapper `scripts/run_fit4488.py`).

For A2 and A3, alpha=0.5 is selected based on PairF1 on the same 653 evaluation papers. These results are therefore exploratory and should not be interpreted as independent estimates of performance.

#### 4. Clustering only the 653 evaluation papers

In this experiment, I use only the 653 evaluation papers **both to fit and to evaluate** (fit = eval). I represent each paper using its **abstract and conclusion**.

I also try different ways of combining these two text sections. I then use UMAP to reduce the dimensionality of the embeddings and HDBSCAN to form clusters.

The results are saved in `artifacts/extension_653/` (shipped CSV from a prior restricted-fit run; not re-executed by the assignment1 GPU runners).

Setup (4) uses results from a prior restricted-fit experiment. The exact UMAP/HDBSCAN configuration could not be fully recovered from the available artifacts, so this setup should be treated as a separate exploratory experiment rather than a controlled comparison with setups (1)-(3). The abstract+conclusion concatenation variant was selected based on PairF1 on the same 653 evaluation papers.

#### 5. Same A3 representation with agglomerative clustering (full 653 coverage)

I keep the A3 title-abstract similarity representation from setup (2) (SPECTER2, alpha=0.1) and apply the same UMAP projection based on the precomputed distance matrix D=1-S. I then replace HDBSCAN with Ward agglomerative clustering (K=30) on the resulting UMAP coordinates.

Unlike HDBSCAN, this assigns every paper to a cluster, so evaluation uses **all 653** papers (Coverage = 1.0). K=30 was selected as the best PairF1 among an ad-hoc grid {30, 54, 80} on the same evaluation set. Both alpha=0.1 and K=30 were selected using these evaluation labels, so this is an exploratory configuration rather than an independent estimate of performance.

Results: `artifacts/assignment1_a3_agglo/results/a3_agglo_headline.csv`.

#### Hyperparameter selection

For setups **(1)-(3)**, UMAP/HDBSCAN settings are **fixed in config** (`n_neighbors=40`, `n_components=5`, `min_cluster_size=25`, `min_samples=5`, `random_state=42`) and are **not** chosen using the 653 SToP labels.

Setup **(4)** uses results from a prior restricted-fit experiment. The exact UMAP/HDBSCAN configuration could not be fully recovered from the available artifacts. These settings were not re-tuned in the assignment1 title-fusion/fit4488 pass, and the abs+concl concatenation variant was selected based on PairF1 on the same 653 evaluation papers. Therefore, this setup should be treated as exploratory rather than as a controlled comparison with setups (1)-(3).

For A2 and A3 in setup **(2)**, and A2 and A3 in setup **(3)**, the full alpha grids are stored in the result CSVs. The reported alpha values were selected based on PairF1 on the same 653 evaluation papers. These results are therefore exploratory and should not be interpreted as independent estimates of performance.

Setup **(5)** reuses alpha=0.1 from setup (2) and selects Ward K from the grid {30, 54, 80} using PairF1 on the same evaluation set. Consequently, both the representation weight and the number of clusters are subject to evaluation-set selection bias.


## 2.2 Discussion of results

### Table 2. Alignment with the SToP taxonomy (2020-2021 subset, 653 papers)

The original benchmark uses method-specific assignment protocols. In particular, ACC relies on hard HDBSCAN assignments and leaves noise documents uncovered, whereas LDA, NMF, and BERTopic provide soft topic assignments. Thus, Table 2 should be interpreted in the context of each method's assignment and coverage protocol.

| Method | Purity | LRAP | nDCG@1 | V-measure | PairF1 | reproduced | Purity_repro | LRAP_repro | nDCG@1_repro | V_repro | PairF1_repro |
|---|---:|---:|---:|---:|---:|:---:|---:|---:|---:|---:|---:|
| LDA | 0.245 | 0.508 | 0.329 | 0.278 | 0.170 | True | 0.245 | 0.510 | 0.331 | 0.270 | 0.167 |
| NMF | 0.294 | 0.585 | 0.445 | 0.324 | 0.186 | True | 0.294 | 0.585 | 0.445 | 0.324 | 0.186 |
| BERTopicSPECTER2 | 0.324 | 0.651 | 0.515 | 0.469 | 0.289 | True | 0.318 | 0.655 | 0.520 | 0.471 | 0.290 |
| BERTopicMPNet | 0.327 | 0.651 | 0.509 | 0.457 | 0.234 | True | 0.326 | 0.656 | 0.516 | 0.463 | 0.249 |
| SentSPECTER | 0.654 | 0.668 | 0.540 | 0.532 | 0.268 | True | 0.654 | 0.668 | 0.540 | 0.532 | 0.268 |
| SentSPECTERtop30 | 0.472 | 0.637 | 0.512 | 0.471 | 0.365 | True | 0.472 | 0.637 | 0.512 | 0.471 | 0.365 |
| ACC (paper) | 0.689 | 0.674 | 0.580 | 0.571 | 0.361 | True | 0.689 | 0.674 | 0.580 | 0.570 | 0.361 |
| ACCtop30 | 0.618 | 0.694 | 0.596 | 0.552 | 0.403 | True | 0.618 | 0.693 | 0.596 | 0.552 | 0.403 |



### My methods

Headline SPECTER2 rows (same five Table 2 metrics). Coverage = fraction of eval papers assigned to a non-noise cluster; HDBSCAN noise fraction = noise fraction on the **fit** set (NaN for agglomerative setup (5)).

| Setup | Method | Purity | LRAP | nDCG@1 | V-measure | PairF1 | Coverage | HDBSCAN noise fraction |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| (1) 6517->653 | SPECTER2 R0 full abstract | 0.666 | 0.695 | 0.632 | 0.630 | **0.525** | 0.772 | 0.268 |
| (2) 6517->653 | SPECTER2 A3 similarity alpha=0.1 | 0.695 | 0.733 | 0.666 | 0.668 | **0.557** | 0.747 | 0.270 |
| (2') 6517->653 | SPECTER2 A1 title||abs concat | 0.729 | 0.729 | 0.665 | 0.655 | **0.549** | 0.778 | 0.238 |
| (3) 4488->653 | SPECTER2 R0 abstract | 0.620 | 0.708 | 0.638 | 0.605 | **0.512** | 0.806 | 0.229 |
| (3) 4488->653 | SPECTER2 title T0 | 0.614 | 0.659 | 0.585 | 0.577 | **0.451** | 0.723 | 0.320 |
| (3) 4488->653 | SPECTER2 A1 title||abs concat | 0.668 | 0.711 | 0.627 | 0.615 | **0.425** | 0.789 | 0.250 |
| (3) 4488->653 | SPECTER2 A2 emb-weight alpha=0.5 | 0.674 | 0.738 | 0.676 | 0.642 | **0.496** | 0.750 | 0.301 |
| (3) 4488->653 | SPECTER2 A3 similarity alpha=0.5 | 0.690 | 0.735 | 0.672 | 0.629 | **0.478** | 0.798 | 0.225 |
| (4) 653->653 | SPECTER2 abs+concl concat | 0.627 | 0.726 | 0.628 | 0.594 | **0.508** | 0.812 | 0.188 |
| (5) 6517->653 | SPECTER2 A3 + UMAP+agglo-ward K=30 (full 653 coverage) | 0.518 | 0.635 | 0.538 | 0.533 | **0.405** | **1.000** | NaN |

(2) picks alpha on the eval set (exploratory). (5) uses the same A3 alpha=0.1 features as (2) but Ward agglomerative clustering; **Coverage=1.0 (all 653 eval papers)**. K=30 is eval-selected (exploratory).
 (2') is the fixed text-concat variant without alpha search. On setup (3), abstract R0 is the best PairF1 among the repeated title-abstract variants (title fusion does not transfer from 6517).

### What the metrics capture

- **Purity** - fraction of papers whose cluster majority matches a SToP label (cluster -> label).
- **LRAP** - ranking quality of soft cluster scores vs multi-label SToP topics.
- **nDCG@1** - quality of the top-ranked cluster assignment per paper.
- **V-measure** - harmonic mean of homogeneity and completeness vs SToP.
- **PairF1** - pairwise agreement: same-cluster pairs vs same-SToP-topic pairs.
- **Coverage** - fraction of eval papers assigned to a cluster.
- **HDBSCAN noise fraction** - fraction of the fit set labeled noise by HDBSCAN (not applicable to agglomerative setup (5)).

For HDBSCAN, LRAP, nDCG@1, V-measure, and PairF1 are computed only on evaluation papers assigned to non-noise clusters. Purity is computed using all evaluation papers, with noise papers having zero cluster membership. For agglomerative clustering, all metrics are computed on all 653 evaluation papers because every paper receives a cluster. Therefore, the scores from these two clustering protocols are not directly comparable without accounting for their different evaluation coverage.

### Fit vs eval (transductive)

- Setups **(1)** and **(2):** fit on **6,517** papers; eval 653 is a subset of the fit set.
- Setup **(3):** fit on **4,488** ACC paper IDs; eval 653 is again a subset of the fit set.
- Setup **(4):** fit = eval = **653** (clusters are built on the evaluation papers themselves).
- Setup **(5)** is exploratory: both the A3 representation weight (alpha = 0.1, selected in setup (2)) and the number of clusters (K = 30) were selected using the same evaluation set. Its score should not be interpreted as an independent estimate of generalization performance.


**Takeaways:** title+abstract helps on the full corpus (best PairF1 **0.557** with HDBSCAN); replacing HDBSCAN by Ward agglomerative on the same A3 features gives full 653 coverage but lower PairF1 (**0.405**); on 4488, abstract-only is more robust (**0.512**); on 653, adding the conclusion helps (**0.508** vs abs-only 0.446).



#### Baseline reproduction

The reproduced baseline results are generally close to the scores reported in the original paper. NMF, SentSPECTER, and SentSPECTERtop30 are reproduced exactly or almost exactly. The differences for LDA, BERTopicSPECTER2, BERTopicMPNet, and ACC are also relatively small. This provides a reasonable reference point for comparing the proposed methods with the original Table 2.

#### Abstract representations and window aggregation

I first compare different ways of representing abstracts on the full collection of 6,517 papers. SPECTER2 with the full abstract encoded as a single input (R0) achieves a PairF1 of 0.525.

Splitting abstracts into windows and aggregating their embeddings (R2 and R3) does not improve the SPECTER2 result: both variants achieve approximately 0.51 PairF1. Thus, in this experiment, processing the full abstract as a single document representation is more effective than averaging representations of separate windows.

The effect of window aggregation depends on the encoder. For SemCSE, token-weighted aggregation improves PairF1 from approximately 0.34 to 0.41, although it remains below the SPECTER2 results. Qwen produces competitive Purity scores but does not achieve similarly strong PairF1. This illustrates that a representation can produce relatively pure clusters without necessarily grouping all papers with the same SToP topics effectively.

The Late MaxSim similarity-based aggregation was also tested, but it did not produce a useful clustering result. Overall, the experiments do not show that window-based representations are consistently better than encoding the full abstract.

#### Combining titles and abstracts

Adding the title to the abstract improves the results on the 6,517-paper collection.

The SPECTER2 abstract-only baseline achieves 0.525 PairF1. Concatenating the title and abstract before encoding (A1) increases it to 0.549. Combining title and abstract similarity matrices (A3) with alpha = 0.1 gives the highest PairF1 among the reported experiments: 0.557.

The improvement is also visible in the other metrics. Compared with the abstract-only representation, A3 increases Purity from 0.666 to 0.695, LRAP from 0.695 to 0.733, nDCG@1 from 0.632 to 0.666, and V-measure from 0.630 to 0.668.

However, A1 and A3 have slightly different strengths. A1 achieves a higher Purity of 0.729, while A3 performs slightly better in LRAP, nDCG@1, V-measure, and PairF1. Therefore, A3 is not strictly better according to every metric, even though it gives the best PairF1.

The A3 result with alpha = 0.1 should be interpreted as exploratory because alpha was selected using the evaluation results. It is not an independent estimate of performance. A1 is a useful complementary result because it combines title and abstract without requiring this alpha selection.

#### Effect of the fitting collection

To investigate how the size and composition of the fitting collection affect the results, I also run the abstract-only SPECTER2 experiment on the 4,488 papers used by the ACC baseline.

This setup achieves Purity = 0.620, LRAP = 0.708, nDCG@1 = 0.638, V-measure = 0.605, and PairF1 = 0.512. Its PairF1 is slightly lower than the 0.525 obtained when fitting on all 6,517 papers.

This result shows that the choice of fitting collection affects clustering quality. In particular, the results obtained on 6,517 papers should not be directly assumed to hold for the smaller ACC-matched collection. The title-and-abstract improvements observed on the full collection should therefore be interpreted within that experimental setup rather than treated as a universal improvement.

The abstract-only result on 4,488 papers also exceeds the published ACCtop30 PairF1 of 0.403. However, this comparison should be interpreted with care: the methods operate on different representations and clustering units, so it does not establish that paper-level clustering is generally superior to claim-level clustering.

#### Abstract and conclusion on the restricted evaluation set

In a separate experiment, I fit the clustering model using only the 653 evaluation papers and compare abstract-only input with a concatenation of the abstract and conclusion.

Adding the conclusion increases PairF1 from 0.446 to 0.508. This suggests that conclusions can provide useful information about a paper's research topic beyond what is available in its abstract.

However, this experiment uses a different fitting setup from the full-collection experiments: the clusters are built using only the evaluation papers themselves. It should therefore be treated as a separate, exploratory experiment, not as a directly comparable improvement over the 6,517-paper setup.

#### Dual-clustering experiments

I additionally investigate whether separately clustering titles and abstracts and then combining the resulting cluster information can improve the final clustering.

The tested aggregation strategies include soft membership concatenation, co-association similarity, and a third clustering based on embeddings and cluster-membership features. None of these approaches improves upon the SPECTER2 abstract-only result on the full collection. The best aggregation, soft_concat, achieves a PairF1 of approximately 0.323.

Thus, in the current implementation, combining two independently produced clusterings is less effective than directly clustering the abstract embeddings. This suggests that errors in the initial title and abstract clusterings may propagate into the aggregation stage rather than being corrected by it.

#### Overall findings

Among the reported main experiments, SPECTER2 with title-abstract similarity fusion (A3, alpha = 0.1) achieves the highest PairF1 of 0.557 on the 6,517-paper fitting collection. Simple title-abstract concatenation (A1) also improves over abstract-only encoding and achieves the highest Purity among these variants.

Window aggregation does not consistently improve abstract representations, and the dual-clustering approaches do not outperform the abstract-only baseline. The restricted 653-paper experiment indicates that conclusions can provide additional topical information, although its fitting setup is different from the full-collection experiments.

Overall, the results suggest that the choice of input text and its representation has a substantial effect on paper clustering quality. In particular, incorporating title information is promising, but its benefit should be evaluated separately for each fitting collection and without selecting hyperparameters on the evaluation labels.

#### Agglomerative clustering on A3 (full coverage)

As a clusterer check on the best A3 representation, I replaced HDBSCAN with Ward agglomerative clustering (K=30) after the same UMAP step. This assigns every paper to a cluster, so metrics are computed on **all 653** evaluation papers (Coverage = 1.0). PairF1 drops to 0.405 versus 0.557 for UMAP+HDBSCAN on the same features, so the stronger headline result remains the HDBSCAN variant despite its lower coverage.


# 3. Code

## 3.1 Requirements

In [16]:
# From the repository root:
# uv sync

from pathlib import Path
import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
ROOT = Path('..').resolve() if not Path('artifacts').exists() else Path('.').resolve()
if not (ROOT / 'artifacts').exists():
    ROOT = Path.cwd().parent



## 3.2 Data & artifact paths

In [17]:
# Data (shipped in the repo / caches):
#   papers 6517: data/claims_sources/emnlp_full/papers.csv
#   ACC 4488 IDs: artifacts/baseline/acc_clusters_emnlp.csv
#   eval 653:     artifacts/extension_full656/author_table2_eval_papers.csv
#   SToP labels:  data/external/STop_topic_classification_dataset_for_scientific_papers.csv

paths = {
    'windows_6517': ROOT / 'artifacts/assignment1/results/stage_ab_results.csv',
    'title_fusion_6517': ROOT / 'artifacts/assignment1_title_fusion/results/title_fusion_results.csv',
    'fit4488': ROOT / 'artifacts/assignment1_fit4488/results/fit4488_results.csv',
    'fair653': ROOT / 'artifacts/extension_653/fair653_all_methods.csv',
    'table2_pub': ROOT / 'artifacts/assignment1/results/table2_published_original.csv',
    'table2_repro': ROOT / 'artifacts/assignment1/results/table2_reproduced_from_artifacts.csv',
    'a3_agglo': ROOT / 'artifacts/assignment1_a3_agglo/results/a3_agglo_headline.csv',
}
for k, p in paths.items():
    print(f'{k}: exists={p.exists()}  ->  {p.relative_to(ROOT)}')


windows_6517: exists=True  ->  artifacts/assignment1/results/stage_ab_results.csv
title_fusion_6517: exists=True  ->  artifacts/assignment1_title_fusion/results/title_fusion_results.csv
fit4488: exists=True  ->  artifacts/assignment1_fit4488/results/fit4488_results.csv
fair653: exists=True  ->  artifacts/extension_653/fair653_all_methods.csv
table2_pub: exists=True  ->  artifacts/assignment1/results/table2_published_original.csv
table2_repro: exists=True  ->  artifacts/assignment1/results/table2_reproduced_from_artifacts.csv


## 3.3 How to reproduce (commands)

Setup numbers match section 2.1.

```bash
# (1) full collection, abstracts only - R0/R2/R3 @6517
uv run python artifacts/assignment1/pipeline/run_workstream_a.py --stage ab --encoders specter2,semcse,qwen

# (2) full collection, titles + abstracts - T0/A1/A2/A3 @6517
uv run python artifacts/assignment1_title_fusion/run_title_fusion.py

# (3) ACC-matched collection - T0/A1/A2/A3 (+ R0) @4488
uv run python artifacts/assignment1_fit4488/run_fit4488.py --encoders specter2,semcse,qwen
# equivalent wrapper: uv run python scripts/run_fit4488.py

# (4) eval-only 653 abs+concl - shipped results only (prior extension; no GPU re-runner here):
#     artifacts/extension_653/fair653_all_methods.csv

# (5) A3 + UMAP + agglomerative ward (full 653 coverage)
#     requires setup (2) similarity cache specter2_S_late_a0.1.npy
uv run python artifacts/assignment1_a3_agglo/run_a3_agglo.py --alpha 0.1 --k 30

# (optional appendix) dual clustering
# uv run python artifacts/assignment1_dual_cluster/run_dual_cluster.py
# uv run python artifacts/assignment1_dual_cluster/run_feat_third_cluster.py
```

Method map: `artifacts/assignment1/README.md`.


## 3.4 Load & show headline results

In [ ]:
pub = pd.read_csv(paths['table2_pub'])
repro = pd.read_csv(paths['table2_repro'])

# --- Table 2: paper + reproduced ---
m = pub.merge(repro, on='method', suffixes=('', '_repro'))
table2 = pd.DataFrame({
    'Method': m['method'],
    'Purity': m['Purity'],
    'LRAP': m['LRAP'],
    'nDCG@1': m['nDCG@1'],
    'V-measure': m['V_measure'],
    'PairF1': m['PairF1'],
    'reproduced': True,
    'Purity_repro': m['Purity_repro'],
    'LRAP_repro': m['LRAP_repro'],
    'nDCG@1_repro': m['nDCG@1_repro'],
    'V_repro': m['V_measure_repro'],
    'PairF1_repro': m['PairF1_repro'],
})
print('Table 2 - paper metrics + reproduced:')
display(table2.style.format({
    'Purity': '{:.3f}', 'LRAP': '{:.3f}', 'nDCG@1': '{:.3f}', 'V-measure': '{:.3f}', 'PairF1': '{:.3f}',
    'Purity_repro': '{:.3f}', 'LRAP_repro': '{:.3f}', 'nDCG@1_repro': '{:.3f}',
    'V_repro': '{:.3f}', 'PairF1_repro': '{:.3f}',
}))

# --- My methods (order matches section 2.1) ---
fair = pd.read_csv(paths['fair653'])
ab = pd.read_csv(paths['windows_6517'])
tf = pd.read_csv(paths['title_fusion_6517'])
f4 = pd.read_csv(paths['fit4488'])

def _noise(r):
    if 'noise_fraction' in r.index and pd.notna(r.get('noise_fraction')):
        return float(r['noise_fraction'])
    return float(r['noise_frac'])

def _row(setup, name, r, lrap='LRAP', ndcg='nDCG@1', v='V_measure'):
    return {
        'Setup': setup,
        'Method': name,
        'Purity': float(r['Purity']),
        'LRAP': float(r[lrap]),
        'nDCG@1': float(r[ndcg]),
        'V-measure': float(r[v]),
        'PairF1': float(r['PairF1']),
        'Coverage': float(r['Coverage']),
        'HDBSCAN noise fraction': _noise(r),
    }

ours_rows = []

# (1) abstracts @6517
r1 = ab[ab.method == 'specter2|R0_full|umap_hdbscan'].iloc[0]
ours_rows.append(_row('(1) 6517->653', 'SPECTER2 R0 full abstract', r1,
                      lrap='LRAP_mean', ndcg='nDCG@1_mean'))

# (2) title+abs @6517
sp = tf[tf.encoder == 'specter2']
r2 = sp[(sp.representation == 'similarity_late_A3') & (sp.alpha == 0.1)].iloc[0]
ours_rows.append(_row('(2) 6517->653', 'SPECTER2 A3 similarity alpha=0.1', r2))
r2b = sp[sp.representation == 'text_concat_A1'].iloc[0]
ours_rows.append(_row("(2') 6517->653", 'SPECTER2 A1 title||abs concat', r2b))

# (3) @4488 - same families as methodology
sp4 = f4[f4.encoder == 'specter2']
ours_rows.append(_row('(3) 4488->653', 'SPECTER2 R0 abstract',
                      sp4[sp4.representation == 'abstract_R0'].iloc[0]))
ours_rows.append(_row('(3) 4488->653', 'SPECTER2 title T0',
                      sp4[sp4.representation == 'title_T0'].iloc[0]))
ours_rows.append(_row('(3) 4488->653', 'SPECTER2 A1 title||abs concat',
                      sp4[sp4.representation == 'text_concat_A1'].iloc[0]))
ours_rows.append(_row('(3) 4488->653', 'SPECTER2 A2 emb-weight alpha=0.5',
                      sp4[(sp4.representation == 'embedding_weighted_A2') & (sp4.alpha == 0.5)].iloc[0]))
ours_rows.append(_row('(3) 4488->653', 'SPECTER2 A3 similarity alpha=0.5',
                      sp4[(sp4.representation == 'similarity_late_A3') & (sp4.alpha == 0.5)].iloc[0]))

# (4) @653 abs+concl
r4 = fair[fair.method == 'OURS_abs+concl_concat'].iloc[0]
ours_rows.append(_row('(4) 653->653', 'SPECTER2 abs+concl concat', r4,
                      lrap='LRAP_mean', ndcg='nDCG@1_mean'))


# (5) A3 + UMAP + agglomerative ward - full coverage of all 653
r5 = pd.read_csv(paths['a3_agglo']).iloc[0]
ours_rows.append({
    'Setup': '(5) 6517->653',
    'Method': 'SPECTER2 A3 + UMAP+agglo-ward K=30 (full 653 coverage)',
    'Purity': float(r5['Purity']),
    'LRAP': float(r5['LRAP']),
    'nDCG@1': float(r5['nDCG@1']),
    'V-measure': float(r5['V_measure']),
    'PairF1': float(r5['PairF1']),
    'Coverage': float(r5['Coverage']),
    'HDBSCAN noise fraction': float('nan'),
})

ours = pd.DataFrame(ours_rows)
print('\nMy methods - Table 2 metrics + Coverage/HDBSCAN noise fraction:')
display(ours.style.format({
    'Purity': '{:.3f}', 'LRAP': '{:.3f}', 'nDCG@1': '{:.3f}',
    'V-measure': '{:.3f}', 'PairF1': '{:.3f}',
    'Coverage': '{:.3f}', 'HDBSCAN noise fraction': '{:.3f}',
}))


## 3.5 Eval subset fit check (regimes A & B)

In [ ]:
eval_ids = set(pd.read_csv(ROOT/'artifacts/extension_full656/author_table2_eval_papers.csv').paper_id.astype(str))
fit_A = set(pd.read_csv(ROOT/'data/claims_sources/emnlp_full/papers.csv').paper_id.astype(str))
fit_B = set(pd.read_csv(ROOT/'artifacts/baseline/acc_clusters_emnlp.csv').paper_id.astype(str).unique())
assert len(eval_ids)==653 and len(fit_A)==6517 and len(fit_B)==4488
assert eval_ids.issubset(fit_A)   # setups (1)-(2): eval subset of 6517 fit
assert eval_ids.issubset(fit_B)   # setup (3): eval subset of 4488 fit
# setup (4): fit == eval == 653
fit4 = pd.read_csv(ROOT / 'artifacts/extension_653/papers.csv')
fit4_ids = fit4.paper_id.astype(str)

assert len(fit4) == 653
assert fit4_ids.nunique() == 653
assert set(fit4_ids) == eval_ids
print('Setup (1)-(2): eval subset of fit_6517 OK')
print('Setup (3): eval subset of fit_4488 OK')
print('Setup (4): fit = eval = 653 (restricted)')


### Conclusion and limitations

In this work, I investigated several approaches to clustering NLP research papers using pretrained text embeddings. I compared abstract-only representations, window-based aggregation, title-abstract fusion, and combinations of independently produced clusterings.

The strongest result among our proposed methods was obtained with SPECTER2 and title-abstract similarity fusion (A3), which achieved a PairF1 of 0.557 when clustering the full collection of 6,517 papers. Concatenating titles and abstracts before encoding (A1) also improved over the abstract-only baseline. In contrast, window aggregation and dual-clustering approaches did not provide consistent improvements. On the 4,488-paper ACC-matched set, abstract R0 remained stronger than title-abstract fusion. Replacing HDBSCAN with Ward agglomerative clustering on the same A3 features yields full coverage of all 653 eval papers but a lower PairF1 (0.405).

The experiments have several limitations.

First, the evaluation is **transductive**. In setups (1)-(3), the 653 evaluation papers are included in the fitting collection (6517 and 4488 respectively). In setup (4), clusters are fit on the evaluation papers themselves (fit = eval = 653). These results measure alignment with SToP labels inside the given collection rather than generalization to completely unseen papers.

Second, the best A3 configuration on 6517 was selected using the evaluation results. Its score should therefore be treated as exploratory, and an independent evaluation would be needed to estimate performance without this selection bias.

In addition, several configurations were selected using PairF1 on the same evaluation set: alpha values in setups (2) and (3), the abstract+conclusion variant in setup (4), and the number of clusters in setup (5). These results are exploratory and may be optimistic due to evaluation-set selection bias.

Third, the experiments use different fitting collections and, for setup (4), a different UMAP/HDBSCAN scale than setups (1)-(3). Results are not directly interchangeable across setups.

Finally, HDBSCAN can assign papers to noise. Coverage and noise fractions for the headline methods are reported in the My methods table; PairF1 and related scores should be read together with those columns.

Despite these limitations, the experiments show that combining title and abstract information is a promising direction for paper-level clustering on the full corpus, while the effect depends on the encoder and the collection used to construct the clusters.


# Appendix A. Dual clustering (exploratory)

**Not part of the five main experimental setups.** Separate title and abstract clusterings on the 6,517-paper corpus, then try to combine them.

### Method

1. Cluster **titles** alone and **abstracts** alone (same UMAP+HDBSCAN settings as Regime A).
2. Aggregate the two partitions in three ways:
   - **soft_concat** - concatenate soft membership bags from both clusterings and evaluate once.
   - **coassoc** - co-association matrix $S_{ij}=\mathbf{1}[\text{same title cl}]+\mathbf{1}[\text{same abs cl}]$, then UMAP on $D=1-S/2$ -> HDBSCAN.
   - **feat_third** - third clustering on features  
     `[L2(title_emb) | L2(abs_emb) | w*onehot(title_cl) | w*onehot(abs_cl)]`  
     with $w \in \{0.5, 1.0, 2.0\}$.

### Code

| Step | Path |
|---|---|
| Dual cluster + soft_concat / coassoc | `artifacts/assignment1_dual_cluster/run_dual_cluster.py` |
| Third clustering (emb + cluster IDs) | `artifacts/assignment1_dual_cluster/run_feat_third_cluster.py` |
| Cluster labels | `artifacts/assignment1_dual_cluster/clusters/` |
| Metrics CSV | `artifacts/assignment1_dual_cluster/results/dual_cluster_results.csv` |

```bash
uv run python artifacts/assignment1_dual_cluster/run_dual_cluster.py
uv run python artifacts/assignment1_dual_cluster/run_feat_third_cluster.py --encoders specter2,semcse,qwen
```

### Metrics (PairF1 on 653; fit = 6,517)

None of the aggregations beats **abstract-only** SPECTER2 (**0.525**). Best aggregation is soft_concat SPECTER2 (**0.323**); feat_third stays in the same ballpark (~0.30-0.32).

In [20]:
dual = pd.read_csv(ROOT / 'artifacts/assignment1_dual_cluster/results/dual_cluster_results.csv')
# keep headline columns; sort encoder -> method family
cols = ['method', 'aggregation', 'PairF1', 'Purity', 'LRAP', 'nDCG@1', 'V_measure', 'n_clusters']
show = dual[cols].copy()
show = show.sort_values(['method']).reset_index(drop=True)
print('Dual clustering appendix - full metric table:')
display(show.style.format({
    'PairF1': '{:.3f}', 'Purity': '{:.3f}', 'LRAP': '{:.3f}',
    'nDCG@1': '{:.3f}', 'V_measure': '{:.3f}',
}))
print('\nBest PairF1 per aggregation family:')
fam = dual.assign(
    family=dual['aggregation'].fillna('none').astype(str).map(
        lambda a: 'single_view' if a == 'none'
        else ('feat_third' if a.startswith('feat_third') else a)
    )
)
display(
    fam.loc[fam.groupby('family')['PairF1'].idxmax(), cols]
      .sort_values('PairF1', ascending=False)
      .reset_index(drop=True)
      .style.format({'PairF1': '{:.3f}', 'Purity': '{:.3f}', 'LRAP': '{:.3f}',
                     'nDCG@1': '{:.3f}', 'V_measure': '{:.3f}'})
)

Dual clustering appendix - full metric table:


,method,aggregation,PairF1,Purity,LRAP,nDCG@1,V_measure,n_clusters
0,qwen|abstract_cluster,none,0.395,0.694,0.674,0.614,0.593,66
1,qwen|dual_coassoc,coassoc,0.273,0.641,0.591,0.505,0.498,77
2,qwen|dual_feat_third_w0.5,feat_third_w0.5,0.268,0.662,0.584,0.513,0.528,102
3,qwen|dual_feat_third_w1.0,feat_third_w1.0,0.266,0.668,0.581,0.507,0.527,101
4,qwen|dual_feat_third_w2.0,feat_third_w2.0,0.279,0.658,0.583,0.517,0.535,96
5,qwen|dual_soft_concat,soft_concat,0.344,0.666,0.663,0.574,0.534,127
6,qwen|title_cluster,none,0.422,0.621,0.623,0.558,0.553,61
7,semcse|abstract_cluster,none,0.345,0.606,0.592,0.515,0.545,46
8,semcse|dual_coassoc,coassoc,0.183,0.592,0.510,0.422,0.421,65
9,semcse|dual_feat_third_w0.5,feat_third_w0.5,0.176,0.643,0.505,0.409,0.456,80



Best PairF1 per aggregation family:


,method,aggregation,PairF1,Purity,LRAP,nDCG@1,V_measure,n_clusters
0,specter2|abstract_cluster,none,0.525,0.666,0.695,0.632,0.630,53
1,qwen|dual_soft_concat,soft_concat,0.344,0.666,0.663,0.574,0.534,127
2,specter2|dual_feat_third_w2.0,feat_third_w2.0,0.315,0.680,0.628,0.569,0.555,88
3,specter2|dual_coassoc,coassoc,0.302,0.644,0.602,0.521,0.523,73
